# Exercises XP: BERT in Practice

Week 12, Day 6

## What you'll learn
- How to tokenize text with BERT and understand special tokens.
- How to run a pretrained sentiment pipeline.
- How to build custom BERT-based sentiment and NER analyzers.
- How to compare encoder (BERT) versus decoder (GPT) families.
- How BERT supplies retrieval power inside a RAG stack.

## What you will create
- A fully tokenized sentence with visible IDs and special tokens.
- A working sentiment pipeline powered by a fine-tuned DistilBERT model.
- Custom helper classes for sentiment classification and NER.
- A comparison table that contrasts BERT and GPT.
- A written explanation (and a small demo) of how BERT embeddings drive retrieval in RAG.

> Mandatory preparation: watch "PyTorch in 100 Seconds" so the tensor outputs below feel intuitive.

## Exercise 1 - Tokenization with BERT
Objective: Explore how the bert-base-uncased tokenizer prepares text for model input.

In [1]:
# Setup: install dependencies if they are missing in your environment (already available in Colab).
%pip install -q transformers torch
import os
os.environ["USE_TF"] = "0"  # use the PyTorch backend only (avoids TensorFlow / Keras 3 conflicts)
import warnings
warnings.filterwarnings("ignore")
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()

Note: you may need to restart the kernel to use updated packages.


In [2]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

sample_sentence = "BERT reads sentences in both directions, unbelievably!"
print(sample_sentence)
print("Basic tokenization (no special tokens):", tokenizer.tokenize(sample_sentence))

BERT reads sentences in both directions, unbelievably!
Basic tokenization (no special tokens): ['bert', 'reads', 'sentences', 'in', 'both', 'directions', ',', 'un', '##bel', '##ie', '##va', '##bly', '!']


In [3]:
encoding = tokenizer(
    sample_sentence,
    add_special_tokens=True,
    padding="max_length",
    truncation=True,
    max_length=24,  # the sentence has 13 tokens + [CLS] + [SEP] = 15, so 24 leaves room and shows the padding
    return_attention_mask=True,
    return_tensors="pt"
)

input_ids = encoding["input_ids"][0].tolist()
tokens = tokenizer.convert_ids_to_tokens(input_ids)
print("index | token        | id    | note")
print("-------------------------------------------")
for idx, (token, token_id) in enumerate(zip(tokens, input_ids)):
    note = "<-- SPECIAL" if token in tokenizer.all_special_tokens else ("(subword)" if token.startswith("##") else "")
    print(f"{idx:>5} | {token:<12} | {token_id:>5} | {note}")

print("\nAttention mask:", encoding["attention_mask"][0].tolist())
special_positions = [(i, tok) for i, tok in enumerate(tokens) if tok in tokenizer.all_special_tokens]
print("Special tokens (index, token):", special_positions)
print("Tensor shapes:", {k: tuple(v.shape) for k, v in encoding.items()})
print("Decoded back:", tokenizer.decode(input_ids, skip_special_tokens=True))

index | token        | id    | note
-------------------------------------------
    0 | [CLS]        |   101 | <-- SPECIAL
    1 | bert         | 14324 | 
    2 | reads        |  9631 | 
    3 | sentences    | 11746 | 
    4 | in           |  1999 | 
    5 | both         |  2119 | 
    6 | directions   |  7826 | 
    7 | ,            |  1010 | 
    8 | un           |  4895 | 
    9 | ##bel        |  8671 | (subword)
   10 | ##ie         |  2666 | (subword)
   11 | ##va         |  3567 | (subword)
   12 | ##bly        |  6321 | (subword)
   13 | !            |   999 | 
   14 | [SEP]        |   102 | <-- SPECIAL
   15 | [PAD]        |     0 | <-- SPECIAL
   16 | [PAD]        |     0 | <-- SPECIAL
   17 | [PAD]        |     0 | <-- SPECIAL
   18 | [PAD]        |     0 | <-- SPECIAL
   19 | [PAD]        |     0 | <-- SPECIAL
   20 | [PAD]        |     0 | <-- SPECIAL
   21 | [PAD]        |     0 | <-- SPECIAL
   22 | [PAD]        |     0 | <-- SPECIAL
   23 | [PAD]        |     0 | <-- SPE

### Exercise 1 reflection

**Deliverables:** the table above lists every token with its ID: **`[CLS]` (id 101)** at position 0, **`[SEP]` (id 102)** right after the last real token, and **`[PAD]` (id 0)** filling the remaining positions up to 24. The uncased tokenizer lower-cases the text, and rare words are split into WordPieces: "unbelievably" becomes `un` + `##bel` + `##ie` + `##va` + `##bly` (the `##` prefix means "continuation of the previous piece"), "BERT" becomes `bert`, and the punctuation marks are separate tokens.

**Padding choice:** the sentence produces 15 tokens with the special tokens, so `max_length=24` is enough to keep the whole sentence (no truncation) while still showing how padding works. In a real batch, we would choose a length that covers most of our sentences (e.g. the 95th percentile of their lengths), or use dynamic padding (`padding=True`) to pad only to the longest sentence of the batch.

- **How `[CLS]` and `[SEP]` behave inside the encoder:** they are ordinary tokens for the Transformer: they have their own embeddings and take part in self-attention like any other token. `[CLS]` is always at position 0; because it attends to all the tokens in both directions, its final hidden state becomes a summary of the whole input, and classification heads (e.g. sentiment) are trained on top of it. `[SEP]` marks the end of a segment; with sentence pairs it separates sentence A from sentence B (together with the `token_type_ids`), so the model knows where each sentence starts and ends.
- **How the attention mask hides padding:** the mask has 1 for real tokens and 0 for `[PAD]` positions. Inside each self-attention layer, the scores towards masked positions get a very large negative value (≈ −∞) added before the softmax, so their attention weights become 0: no real token "looks at" padding, and the padding does not change the representations or the prediction. This lets us batch sentences of different lengths together.

## Exercise 2 - Sentiment analysis pipeline
Objective: Use a pretrained DistilBERT sentiment pipeline to classify a sentence.

In [4]:
from transformers import pipeline

sentiment_pipeline = pipeline(
    task="sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

sentence = "The new update made the app so much faster, I absolutely love it!"
prediction = sentiment_pipeline(sentence)
prediction

[{'label': 'POSITIVE', 'score': 0.9998602867126465}]

In [5]:
# A few more sentences to probe the model's behaviour
for s in ["The movie was okay, nothing special.",
          "I wouldn't say the service was bad.",
          "What a fantastic waste of two hours."]:
    print(f"{s!r:<45} -> {sentiment_pipeline(s)[0]}")

'The movie was okay, nothing special.'        -> {'label': 'NEGATIVE', 'score': 0.992511510848999}
"I wouldn't say the service was bad."         -> {'label': 'POSITIVE', 'score': 0.9845644235610962}
'What a fantastic waste of two hours.'        -> {'label': 'NEGATIVE', 'score': 0.9995824694633484}


### Exercise 2 reflection

- **Sentence tested:** *"The new update made the app so much faster, I absolutely love it!"* → **POSITIVE** with a score of **0.9999**.
- **Does it match my expectation?** Yes: the sentence contains strong positive words ("faster", "absolutely love") and no negation, so a clear POSITIVE was expected. The extra examples show the limits of a model fine-tuned on movie reviews (SST-2) with only two classes: *"I wouldn't say the service was bad"* is correctly recognised as POSITIVE (the model handles the negation of "bad"), the sarcastic *"What a fantastic waste of two hours"* is correctly NEGATIVE, but the lukewarm *"The movie was okay, nothing special"* is pushed to NEGATIVE at 99%: the model has no NEUTRAL class, so mixed or neutral texts are forced into one of the two labels.
- **How confident is the model and what does the score tell us?** The score is the **softmax probability** of the predicted class (here 99.99%). It tells us how sure the model is, not how *positive* the sentence is, and it is often over-confident: even neutral sentences get scores above 0.98. For decisions we should therefore treat very high scores with some caution, and if needed calibrate the probabilities or add a neutral threshold (e.g. call a text neutral when the score is below 0.9).

## Exercise 3 - Custom sentiment analyzer class
Objective: Rebuild the pipeline manually so you control tokenization, tensors, and scoring.

In [6]:
import re
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from typing import Dict

class BERTSentimentAnalyzer:
    def __init__(self, model_name: str = "distilbert-base-uncased-finetuned-sst-2-english", max_length: int = 128):
        '''Load the tokenizer/model and move the model to the proper device.'''
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.max_length = max_length
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name).to(self.device)
        self.model.eval()                                # inference mode (disables dropout)
        self.id2label = self.model.config.id2label       # {0: 'NEGATIVE', 1: 'POSITIVE'}

    def preprocess(self, text: str) -> Dict[str, torch.Tensor]:
        '''Clean the text, tokenize, and return tensors ready for inference.'''
        text = re.sub(r"<[^>]+>", " ", text)             # remove HTML tags
        text = re.sub(r"http\S+", " ", text)             # remove URLs
        text = re.sub(r"\s+", " ", text).strip()         # normalise whitespace
        if not text:
            raise ValueError("Empty text after cleaning")
        encoded = self.tokenizer(text, truncation=True, max_length=self.max_length,
                                 padding=True, return_tensors="pt")
        return {k: v.to(self.device) for k, v in encoded.items()}

    def predict(self, text: str) -> Dict[str, float]:
        '''Run a forward pass, apply softmax, and return a label plus probability.'''
        inputs = self.preprocess(text)
        with torch.no_grad():
            logits = self.model(**inputs).logits         # shape (1, 2): raw scores
        probs = torch.softmax(logits, dim=-1)[0]         # probabilities that sum to 1
        best = int(torch.argmax(probs))
        return {
            "label": self.id2label[best],
            "probability": round(float(probs[best]), 4),
            "probabilities": {self.id2label[i]: round(float(p), 4) for i, p in enumerate(probs)},
        }

In [7]:
analyzer = BERTSentimentAnalyzer()
samples = [
    "This is the best purchase I have made all year, highly recommended!",
    "Terrible customer service, I waited two hours and nobody helped me.",
    "<p>The food was cold</p> but the staff were really friendly. https://example.com",
    "It's fine.",
]
for text in samples:
    print(text)
    print("   ->", analyzer.predict(text))

# Sanity check: same result as the pipeline
print("\nPipeline:", sentiment_pipeline(sentence)[0], "| Custom:", analyzer.predict(sentence)["label"], analyzer.predict(sentence)["probability"])

This is the best purchase I have made all year, highly recommended!
   -> {'label': 'POSITIVE', 'probability': 0.9999, 'probabilities': {'NEGATIVE': 0.0001, 'POSITIVE': 0.9999}}
Terrible customer service, I waited two hours and nobody helped me.
   -> {'label': 'NEGATIVE', 'probability': 0.9997, 'probabilities': {'NEGATIVE': 0.9997, 'POSITIVE': 0.0003}}
<p>The food was cold</p> but the staff were really friendly. https://example.com
   -> {'label': 'POSITIVE', 'probability': 0.9993, 'probabilities': {'NEGATIVE': 0.0007, 'POSITIVE': 0.9993}}
It's fine.
   -> {'label': 'POSITIVE', 'probability': 0.9999, 'probabilities': {'NEGATIVE': 0.0001, 'POSITIVE': 0.9999}}

Pipeline: {'label': 'POSITIVE', 'score': 0.9998602867126465} | Custom: POSITIVE 0.9999


**Role of each component:**
- **Tokenizer** (`AutoTokenizer`): turns the cleaned text into `input_ids` and an `attention_mask` (adding `[CLS]`/`[SEP]`, truncating to `max_length`), as PyTorch tensors.
- **Model** (`AutoModelForSequenceClassification`): DistilBERT encoder + a classification head on the `[CLS]` representation; it outputs two **logits** (raw scores) for NEGATIVE and POSITIVE. `model.eval()` and `torch.no_grad()` switch off dropout and gradient computation for fast, deterministic inference.
- **Post-processing:** `softmax` converts the logits into probabilities, `argmax` picks the class, and `id2label` maps the index to a readable label.
- **Preprocessing** is where we gain control compared with the pipeline: here we strip HTML tags and URLs before tokenizing, and we could add language detection, lower-casing rules, batching or a custom threshold. The results match the pipeline exactly, because it runs the same steps internally.

## Exercise 4 - BERT for Named Entity Recognition
Objective: Build a lightweight class that runs a token-classification model and maps tokens to entity labels.

In [8]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
import torch

class BERTNamedEntityRecognizer:
    def __init__(self, model_name: str = "dslim/bert-base-NER"):
        '''Load the tokenizer and model, and detect the available device.'''
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForTokenClassification.from_pretrained(model_name).to(self.device)
        self.model.eval()
        self.id2label = self.model.config.id2label   # e.g. {0: 'O', 1: 'B-MISC', 2: 'I-MISC', 3: 'B-PER', ...}

    def recognize(self, text: str):
        '''Tokenize the text, run the model, map predictions to BIO labels, and merge word pieces.'''
        enc = self.tokenizer(text, return_offsets_mapping=True, truncation=True, return_tensors="pt")
        offsets = enc.pop("offset_mapping")[0].tolist()
        word_ids = enc.word_ids(0)                  # which word each token belongs to (None for [CLS]/[SEP])
        with torch.no_grad():
            logits = self.model(**{k: v.to(self.device) for k, v in enc.items()}).logits[0]
        probs = torch.softmax(logits, dim=-1)
        pred_ids = probs.argmax(-1).tolist()

        entities, current = [], None
        prev_word = None
        for i, word_id in enumerate(word_ids):
            if word_id is None:                     # special tokens
                continue
            label = self.id2label[pred_ids[i]]
            start, end = offsets[i]
            score = float(probs[i, pred_ids[i]])
            if word_id == prev_word and current is not None:
                # subword piece ("##..."): extend the current word/entity, whatever its own label
                current["end"] = end
                current["scores"].append(score)
            elif label == "O":
                if current: entities.append(current)
                current = None
            else:
                tag, ent_type = label.split("-", 1)
                if tag == "I" and current is not None and current["entity"] == ent_type:
                    current["end"] = end           # continuation of the same entity (next word)
                    current["scores"].append(score)
                else:                               # B- tag (or an I- tag that starts a new entity)
                    if current: entities.append(current)
                    current = {"entity": ent_type, "start": start, "end": end, "scores": [score]}
            prev_word = word_id
        if current:
            entities.append(current)

        return [{"text": text[e["start"]:e["end"]], "entity": e["entity"], "start": e["start"], "end": e["end"],
                 "score": round(sum(e["scores"]) / len(e["scores"]), 4)} for e in entities]

In [9]:
ner = BERTNamedEntityRecognizer()
print("Labels:", ner.id2label)

sample_text = ("Elon Musk announced that Tesla will open a new Gigafactory near Berlin, "
               "while Satya Nadella presented Microsoft's AI plans at a conference in San Francisco.")
entities = ner.recognize(sample_text)
for e in entities:
    print(e)

# Token-level view: how the word pieces were labelled before merging
enc = ner.tokenizer(sample_text, return_tensors="pt")
with torch.no_grad():
    pred = ner.model(**enc).logits.argmax(-1)[0].tolist()
toks = ner.tokenizer.convert_ids_to_tokens(enc["input_ids"][0])
print("\nToken-level predictions:")
print([(t, ner.id2label[p]) for t, p in zip(toks, pred) if ner.id2label[p] != "O"])

# Comparison with the built-in pipeline aggregation
print("\nPipeline (aggregation_strategy='simple'):")
for e in pipeline("ner", model="dslim/bert-base-NER", aggregation_strategy="simple")(sample_text):
    print({k: (round(float(v), 4) if k == "score" else v) for k, v in e.items()})

Labels: {0: 'O', 1: 'B-MISC', 2: 'I-MISC', 3: 'B-PER', 4: 'I-PER', 5: 'B-ORG', 6: 'I-ORG', 7: 'B-LOC', 8: 'I-LOC'}
{'text': 'Elon Musk', 'entity': 'ORG', 'start': 0, 'end': 9, 'score': 0.9698}
{'text': 'Tesla', 'entity': 'PER', 'start': 25, 'end': 30, 'score': 0.8454}
{'text': 'Gigafactory', 'entity': 'ORG', 'start': 47, 'end': 58, 'score': 0.7966}
{'text': 'Berlin', 'entity': 'LOC', 'start': 64, 'end': 70, 'score': 0.9991}
{'text': 'Satya Nadella', 'entity': 'PER', 'start': 78, 'end': 91, 'score': 0.9236}
{'text': 'Microsoft', 'entity': 'ORG', 'start': 102, 'end': 111, 'score': 0.9979}
{'text': 'AI', 'entity': 'MISC', 'start': 114, 'end': 116, 'score': 0.9856}
{'text': 'San Francisco', 'entity': 'LOC', 'start': 142, 'end': 155, 'score': 0.9994}

Token-level predictions:
[('El', 'B-ORG'), ('##on', 'I-ORG'), ('Mu', 'I-ORG'), ('##sk', 'I-ORG'), ('Te', 'B-PER'), ('##sla', 'I-PER'), ('G', 'B-ORG'), ('##iga', 'I-ORG'), ('##fa', 'I-ORG'), ('Berlin', 'B-LOC'), ('Sa', 'B-PER'), ('##ty', 'B-PER

{'entity_group': 'ORG', 'score': 0.9698, 'word': 'Elon Musk', 'start': 0, 'end': 9}
{'entity_group': 'PER', 'score': 0.8454, 'word': 'Tesla', 'start': 25, 'end': 30}
{'entity_group': 'ORG', 'score': 0.8295, 'word': 'Gigafa', 'start': 47, 'end': 53}
{'entity_group': 'LOC', 'score': 0.9991, 'word': 'Berlin', 'start': 64, 'end': 70}
{'entity_group': 'PER', 'score': 0.9996, 'word': 'Sa', 'start': 78, 'end': 80}
{'entity_group': 'PER', 'score': 0.9084, 'word': '##tya Nadella', 'start': 80, 'end': 91}
{'entity_group': 'ORG', 'score': 0.9979, 'word': 'Microsoft', 'start': 102, 'end': 111}
{'entity_group': 'MISC', 'score': 0.9856, 'word': 'AI', 'start': 114, 'end': 116}
{'entity_group': 'LOC', 'score': 0.9994, 'word': 'San Francisco', 'start': 142, 'end': 155}


### Exercise 4 deliverables

**Output format:** `recognize()` returns a list of dictionaries `{text, entity, start, end, score}`, where `start`/`end` are character positions in the original text (so `text[start:end]` gives the entity exactly) and `score` is the average probability of the entity's tokens.

**Results on the sample paragraph:** the model correctly finds **Berlin** and **San Francisco** (LOC), **Satya Nadella** (PER), **Microsoft** (ORG) and **AI** (MISC). It also makes mistakes: **"Elon Musk" is labelled ORG** and **"Tesla" PER** (the two are swapped, probably because they appear together and "Tesla" is also a person's name), and **"Gigafactory"** is labelled ORG, which is debatable. The scores of these doubtful entities are clearly lower (0.80–0.85 against > 0.99 for Berlin or Microsoft), so the score is a useful signal to flag uncertain predictions. A model fine-tuned on more recent or domain-specific text would do better.

**How subword tokens (`##`) are handled:** BERT's WordPiece tokenizer splits rare words into pieces (`Na`, `##dell`, `##a` for "Nadella"; `G`, `##iga`, `##fa`, `##ctory` for "Gigafactory"), and the model predicts a label for **every piece**. As the token-level view shows, these labels can be inconsistent (`Sa` → B-PER, `##ty` → B-PER again). We use the tokenizer's **`word_ids()`** to know which word each piece belongs to: all pieces of the same word are merged into the entity started by the **first piece**, whatever their own labels, and we use the **`offset_mapping`** (character start/end of each token) to rebuild the exact original text, with its original case and without the `##` markers. Consecutive words are joined into one entity when the next word has an `I-` tag of the same type ("Satya" + "Nadella", "San" + "Francisco"). This gives cleaner results than the built-in pipeline with `aggregation_strategy="simple"`, which here cuts "Gigafactory" into "Gigafa" and splits "Satya Nadella" into two entities ("Sa" and "##tya Nadella").

## Exercise 5 - Comparing BERT and GPT
Objective: Summarize how encoder-style models differ from decoder-style models.

| Category | BERT | GPT |
|----------|------|-----|
| Architecture | **Encoder-only** Transformer, **bidirectional** self-attention (each token sees left and right context); pre-trained with masked language modeling (+ NSP originally) | **Decoder-only** Transformer, **unidirectional** (causal mask: each token sees only the previous ones); pre-trained with next-token prediction (causal LM) |
| Primary purpose | **Understanding** text: rich contextual representations of every token and of the whole input | **Generating** text: produce coherent continuations token by token |
| Typical use cases | Text classification (sentiment, spam, intent), named-entity recognition, extractive question answering, semantic search / embeddings for retrieval, reranking | Chatbots and assistants, writing and summarising, translation, code generation, few-shot / zero-shot tasks via prompting |
| Strengths | Deep context understanding from both sides, small and fast (110M params for base), cheap to fine-tune, excellent accuracy on classification/extraction, great embeddings for search | Fluent, flexible generation; one model can do many tasks from a prompt without fine-tuning; scales to very large sizes with strong reasoning and knowledge |
| Weaknesses | Cannot generate free text naturally; needs task-specific fine-tuning and labelled data; limited input length (512 tokens) | Only sees left context, so representations are less suited to some understanding tasks at small sizes; large models are expensive and slow; can hallucinate convincing but false content |

**Reflection.** BERT and GPT share the same building block, the Transformer layer with self-attention, feed-forward networks, residual connections and positional embeddings, and both follow the **pre-train then adapt** recipe on huge unlabeled text corpora. They differ in *which half* of the original Transformer they keep and in their training objective: BERT's bidirectional encoder and masked-token objective make it a powerful *reader* that understands a fixed text, while GPT's causal decoder and next-token objective make it a *writer* that produces text. In practice they are complementary: BERT-style models are the efficient choice for classification, extraction and retrieval, GPT-style models for open-ended generation and conversation, and modern systems such as RAG pipelines (Exercise 6) combine both.

## Exercise 6 - BERT inside Retrieval-Augmented Generation

**What is RAG?** Retrieval-Augmented Generation combines a **retriever**, which finds the documents relevant to a question in an external knowledge base, with a **generator** (an LLM such as GPT), which writes the answer using these documents as context. The model can then answer with up-to-date, private or domain-specific information that it never saw during pre-training, and with fewer hallucinations, because its answer is grounded in real sources that can be cited.

1. **How BERT encodes queries and documents.** In the retrieval component, a BERT-style **bi-encoder** (e.g. Sentence-BERT, DPR, or `all-MiniLM-L6-v2`, a small model of the BERT family) turns each text into a **dense embedding**: the text is tokenized, passed through the encoder, and the token vectors are pooled into one fixed-size vector (the `[CLS]` vector or, more often, the mean of all token vectors), usually normalised to length 1. Documents are first split into chunks (e.g. 200–500 words), and each chunk is embedded **once, offline**; the user's question is embedded with the same model **at query time**. Because the encoder is bidirectional and fine-tuned on question/passage pairs, texts with the same meaning get close vectors even if they use different words ("refund" vs "money back").
2. **How embeddings are stored and searched in a vector database.** All chunk embeddings are stored in a **vector database** (FAISS, Pinecone, Weaviate, Milvus, Chroma, pgvector) together with their text and metadata (source, date, page). At query time, the database computes the **similarity** between the query vector and the stored vectors (cosine similarity or dot product) and returns the **top-k** closest chunks. For millions of documents, it uses **approximate nearest-neighbour** indexes (HNSW graphs, IVF clustering, product quantisation) to answer in milliseconds instead of comparing the query with every vector. Metadata filters (e.g. only documents from 2024) can be combined with the vector search, and a BERT **cross-encoder** can rerank the top results for more precision.
3. **How the retrieved passages are handed to a generative model.** The top-k passages are inserted into a **prompt** together with the question and instructions, e.g. *"Answer the question using only the context below. Cite your sources. If the answer is not in the context, say you don't know. Context: [passage 1] [passage 2] … Question: …"*. The generative model (GPT, Claude, LLaMA…) reads this augmented prompt and writes a fluent answer grounded in the passages; the application can show the sources next to the answer.
4. **Concrete application example.** A **bank's customer-support assistant**: the bank's FAQ, product brochures and internal policies (thousands of pages, updated every week) are chunked and embedded with a BERT-based encoder into a vector database. When a customer asks *"Can I get my money back if my card was charged twice?"*, the retriever finds the "duplicate transactions" and "chargeback policy" passages even though the customer never used those words, and GPT writes a clear answer in the customer's language, citing the policy and its current deadline. Updating the knowledge only requires re-embedding the changed documents, with no retraining of the LLM, and the answers stay faithful to the bank's official rules. The same pattern is used for legal research, medical guidelines, technical documentation search and enterprise knowledge assistants.

Below is a minimal working demo of this pipeline: a BERT-family encoder, an in-memory "vector database" (a matrix of normalised embeddings searched with cosine similarity) and GPT-2 as the generator.

In [10]:
import numpy as np
from transformers import AutoModel

# 1) Knowledge base: small documents (chunks)
documents = [
    "Duplicate transactions: if your card is charged twice for the same purchase, the second charge is refunded automatically within 5 business days.",
    "Chargeback policy: you can dispute a card payment within 60 days of the transaction date by contacting customer support.",
    "Lost or stolen cards can be blocked instantly in the mobile app under Settings > Card > Block card.",
    "International transfers are processed within 1 to 3 business days and cost a flat fee of 5 euros.",
    "Our branches are open Monday to Friday from 9am to 5pm and on Saturday mornings.",
    "The savings account offers an interest rate of 2.5% per year, paid monthly.",
]

# 2) BERT-family encoder (MiniLM, a distilled BERT trained for sentence embeddings) with mean pooling
emb_name = "sentence-transformers/all-MiniLM-L6-v2"
emb_tokenizer = AutoTokenizer.from_pretrained(emb_name)
emb_model = AutoModel.from_pretrained(emb_name).eval()

def embed(texts):
    enc = emb_tokenizer(texts, padding=True, truncation=True, max_length=256, return_tensors="pt")
    with torch.no_grad():
        token_vectors = emb_model(**enc).last_hidden_state            # (n, seq_len, 384)
    mask = enc["attention_mask"].unsqueeze(-1).float()
    vectors = (token_vectors * mask).sum(1) / mask.sum(1)              # mean pooling over real tokens
    return torch.nn.functional.normalize(vectors, dim=-1).numpy()      # unit length -> dot product = cosine

# 3) "Vector database": embeddings computed once and stored
doc_vectors = embed(documents)
print("Vector store:", doc_vectors.shape, "(documents x embedding dimension)")

def retrieve(query, k=2):
    q = embed([query])[0]
    scores = doc_vectors @ q                                           # cosine similarity with every document
    top = np.argsort(-scores)[:k]
    return [(float(scores[i]), documents[i]) for i in top]

query = "Can I get my money back if my card was charged twice?"
retrieved = retrieve(query)
print(f"\nQuery: {query}")
for score, doc in retrieved:
    print(f"  {score:.3f} | {doc}")

Vector store: (6, 384) (documents x embedding dimension)

Query: Can I get my money back if my card was charged twice?
  0.719 | Duplicate transactions: if your card is charged twice for the same purchase, the second charge is refunded automatically within 5 business days.
  0.623 | Chargeback policy: you can dispute a card payment within 60 days of the transaction date by contacting customer support.


In [11]:
# 4) Generation: hand the retrieved passages to a generative model (GPT-2 here, a real system would use a stronger LLM)
from transformers import set_seed
set_seed(0)
generator = pipeline("text-generation", model="gpt2")

context = "\n".join(f"- {doc}" for _, doc in retrieved)
prompt = (f"Use the bank information below to answer the customer.\n"
          f"Information:\n{context}\n\n"
          f"Customer question: {query}\n"
          f"Answer:")
print(prompt)
out = generator(prompt, max_new_tokens=40, do_sample=False, pad_token_id=generator.tokenizer.eos_token_id)[0]["generated_text"]
print("\nGenerated answer:", out[len(prompt):].strip().split("\n")[0])

Use the bank information below to answer the customer.
Information:
- Duplicate transactions: if your card is charged twice for the same purchase, the second charge is refunded automatically within 5 business days.
- Chargeback policy: you can dispute a card payment within 60 days of the transaction date by contacting customer support.

Customer question: Can I get my money back if my card was charged twice?
Answer:



Generated answer: If you have a card that was charged twice, you can claim the refund within 30 days of the transaction date.


**What the demo shows:** the BERT-family encoder retrieves the right passages: the "duplicate transactions" chunk gets the highest cosine similarity (0.72), followed by the "chargeback policy" chunk (0.62), although the question uses different words ("money back", "charged twice"). The retrieved passages are inserted into the prompt and GPT-2 writes an answer. However, GPT-2's answer (*"you can claim the refund within 30 days"*) is **wrong**: the context says the duplicate charge is refunded automatically within **5 business days** (and disputes are possible within 60 days). This illustrates two important points about RAG: good retrieval is necessary but not sufficient, and the **generator must be strong and instruction-tuned** (GPT-4, Claude, LLaMA-Instruct…) to stay faithful to the context. A small, non-instruction-tuned model like GPT-2 still hallucinates. Production systems add a capable LLM, explicit instructions to answer only from the context, source citations, and automatic checks of the answer against the retrieved passages.